# Buổi 4 — Perceptron từ C / I / T

Quy trình của notebook: **Predict → Code → Check → Explain**. Trước khi chạy mỗi lượt, hãy ghi dự đoán vào bảng; sau đó chạy các cell Python độc lập ngay trong notebook.

## 1. Dự đoán trước khi chạy

Với $\eta=0.5$, $\mathbf{w}^{(0)}=(0,0,0,0,0)$ và $b^{(0)}=-0.25$:

| Lượt | Mẫu | net dự đoán | $\hat y$ dự đoán | Có cập nhật? |
|---:|---|---:|---:|---|
| 1 | C |  |  |  |
| 2 | I |  |  |  |
| 3 | T |  |  |  |

## 2. Cài đặt Perceptron ngay trong notebook

Cell Python tiếp theo định nghĩa dữ liệu C/I/T, hàm dự đoán, quy tắc cập nhật và hàm kiểm tra. Notebook có thể chạy độc lập, không cần mở hoặc sửa file bên ngoài.

In [7]:
import numpy as np

LABELS = ("C", "I", "T")
X = np.array(
    [
        [1, 0, 1, 1, 0],  # C
        [0, 0, 0, 0, 1],  # I
        [0, 0, 1, 0, 1],  # T
    ],
    dtype=float,
)
Y = np.array([1, 0, 0], dtype=int)

ETA = 0.5
W0 = np.zeros(X.shape[1], dtype=float)
B0 = -0.25


def predict(x: np.ndarray, w: np.ndarray, b: float) -> tuple[float, int]:
    net = float(np.dot(w, x) + b)
    y_hat = int(net >= 0)
    return net, y_hat


def update(
    x: np.ndarray,
    y: int,
    w: np.ndarray,
    b: float,
    eta: float,
) -> tuple[float, int, int, np.ndarray, float]:
    net, y_hat = predict(x, w, b)
    error = y - y_hat
    w_new = w + eta * error * x
    b_new = b + eta * error
    return net, y_hat, error, w_new, b_new


def train_one_epoch(eta: float = ETA) -> tuple[np.ndarray, float, list[dict[str, object]]]:
    w = W0.copy()
    b = B0
    history: list[dict[str, object]] = []

    for label, x_i, y_i in zip(LABELS, X, Y):
        net, y_hat, error, w, b = update(x_i, int(y_i), w, b, eta)
        history.append(
            {
                "mau": label,
                "net": net,
                "y_hat": y_hat,
                "error": error,
                "w_sau_luot": w.copy(),
                "b_sau_luot": b,
            }
        )

    return w, b, history


def run_checks() -> None:
    net, y_hat = predict(X[0], W0, B0)
    assert np.isclose(net, -0.25)
    assert y_hat == 0

    w_final, b_final, history = train_one_epoch()
    assert len(history) == 3
    assert [row["error"] for row in history] == [1, -1, 0]
    assert np.allclose(w_final, [0.5, 0.0, 0.5, 0.5, -0.5])
    assert np.isclose(b_final, -0.25)


print("Đã định nghĩa Perceptron C/I/T ngay trong notebook.")

Đã định nghĩa Perceptron C/I/T ngay trong notebook.


In [8]:
run_checks()
print('✓ Các kiểm tra Perceptron đã vượt qua.')

✓ Các kiểm tra Perceptron đã vượt qua.


## 3. Đối chiếu một epoch với bảng tính tay

In [14]:
w_final, b_final, history = train_one_epoch()

rows_for_display = []
for turn, row in enumerate(history, start=1):
    rows_for_display.append(
        {
            "Lượt": turn,
            "Mẫu": row["mau"],
            "net": f'{row["net"]:.2f}',
            "y thật": int(Y[turn - 1]),
            "y dự đoán": row["y_hat"],
            "Lỗi": row["error"],
            "Cập nhật?": "Có" if row["error"] != 0 else "Không",
            "w sau lượt": np.array2string(
                row["w_sau_luot"], precision=2, separator=", "
            ),
            "b sau lượt": f'{row["b_sau_luot"]:.2f}',
        }
    )

import pandas as pd

comparison_table = pd.DataFrame(rows_for_display)
display(comparison_table)
print(f'w cuối = {np.array2string(w_final, precision=2, separator=", ")}')
print(f'b cuối = {b_final:.2f}')

,Lượt,Mẫu,net,y thật,y dự đoán,Lỗi,Cập nhật?,w sau lượt,b sau lượt
0,1,C,-0.25,1,0,1,Có,"[0.5, 0. , 0.5, 0.5, 0. ]",0.25
1,2,I,0.25,0,1,-1,Có,"[ 0.5, 0. , 0.5, 0.5, -0.5]",-0.25
2,3,T,-0.25,0,0,0,Không,"[ 0.5, 0. , 0.5, 0.5, -0.5]",-0.25


w cuối = [ 0.5,  0. ,  0.5,  0.5, -0.5]
b cuối = -0.25


**Giải thích bằng lời:** lượt nào làm trọng số thay đổi? Feature đang bật đã kéo trọng số theo hướng nào?

> Viết câu trả lời 3–5 dòng tại đây.

## 4. Thí nghiệm learning rate

Chạy cùng một epoch với ba giá trị $\eta$. Quan sát độ lớn bước cập nhật; không kết luận tốc độ hội tụ chỉ từ một epoch.

In [10]:
for eta in (0.1, 0.5, 1.0):
    w_eta, b_eta, rows_eta = train_one_epoch(eta=eta)
    errors = [row['error'] for row in rows_eta]
    print(f'eta={eta}: errors={errors}, w={w_eta}, b={b_eta:.2f}')

eta=0.1: errors=[1, 0, 0], w=[0.1 0.  0.1 0.1 0. ], b=-0.15
eta=0.5: errors=[1, -1, 0], w=[ 0.5  0.   0.5  0.5 -0.5], b=-0.25
eta=1.0: errors=[1, -1, 0], w=[ 1.  0.  1.  1. -1.], b=-0.25


**Kết luận:** khi $\eta$ tăng, độ lớn mỗi bước cập nhật thay đổi thế nào?

> Viết câu trả lời 2–3 dòng tại đây.

## 5. Mở rộng tự chọn — XOR

XOR có bốn điểm đầu vào và nhãn $y=1$ khi hai bit khác nhau:

| $x_1$ | $x_2$ | $y$ | Ví dụ liên hệ với C / I / T |
|---:|---:|---:|---|
| 0 | 0 | 0 | Không có đặc trưng nào bật → không phải C, giống lớp I/T |
| 0 | 1 | 1 | Chỉ đặc trưng thứ hai bật → thuộc lớp dương, tương tự C |
| 1 | 0 | 1 | Chỉ đặc trưng thứ nhất bật → thuộc lớp dương, tương tự C |
| 1 | 1 | 0 | Cả hai đặc trưng cùng bật → không thuộc lớp dương, giống lớp I/T |

Trong bài C/I/T, Perceptron đang học bài toán nhị phân **C so với không phải C**: mẫu C có nhãn $y=1$, còn mẫu I và T có nhãn $y=0$. Ví dụ:

- Gặp mẫu C: cần dự đoán $\hat y=1$. Nếu dự đoán $0$, sai số là $1$ và trọng số được kéo theo hướng nhận diện C.
- Gặp mẫu I hoặc T: cần dự đoán $\hat y=0$. Nếu dự đoán $1$, sai số là $-1$ và trọng số được kéo theo hướng giảm kích hoạt nhầm.
- Trong bảng XOR, nhãn $1$ đóng vai trò tương tự lớp C; nhãn $0$ đóng vai trò tương tự nhóm I/T. Tuy nhiên, $x_1$ và $x_2$ ở đây chỉ là hai đặc trưng minh họa, không phải năm đặc trưng chữ C/I/T ở phần trên.

Một Perceptron chỉ tạo được một đường biên tuyến tính:

$$
\hat{y}=\mathbb{1}(w_1x_1+w_2x_2+b\ge 0).
$$

Với XOR, hai điểm dương nằm trên hai góc đối diện, còn hai điểm âm nằm ở hai góc còn lại. Không có một đường thẳng nào tách hai nhóm này hoàn toàn, nên dữ liệu **không khả tách tuyến tính**. Vì vậy, khi huấn luyện nhiều epoch, số lỗi có thể thay đổi nhưng không giảm ổn định về $0$.

Ta vẫn dùng đúng quy tắc cập nhật:

$$
\mathbf{w}\leftarrow\mathbf{w}+\eta(y-\hat y)\mathbf{x},\qquad
b\leftarrow b+\eta(y-\hat y).
$$

Mỗi epoch đi qua cả bốn mẫu theo cùng một thứ tự. Hãy quan sát số lỗi sau từng epoch và giải thích vì sao tiếp tục huấn luyện không tạo ra nghiệm phân loại hoàn hảo.

In [11]:
X_XOR = np.array(
    [
        [0.0, 0.0],
        [0.0, 1.0],
        [1.0, 0.0],
        [1.0, 1.0],
    ]
)
Y_XOR = np.array([0, 1, 1, 0])

w_xor = np.zeros(2)
b_xor = 0.0
eta_xor = 0.2
error_counts = []

for epoch in range(1, 11):
    errors = 0
    for x_i, y_i in zip(X_XOR, Y_XOR):
        net, y_hat = predict(x_i, w_xor, b_xor)
        error = int(y_i) - y_hat
        if error != 0:
            errors += 1
        _, _, _, w_xor, b_xor = update(x_i, int(y_i), w_xor, b_xor, eta_xor)
    error_counts.append(errors)
    print(f'epoch {epoch:2d}: errors={errors}, w={w_xor}, b={b_xor:.2f}')

print('Số lỗi không về 0 ổn định vì XOR không khả tách tuyến tính.')

epoch  1: errors=3, w=[-0.2  0. ], b=-0.20
epoch  2: errors=3, w=[-0.2  0. ], b=0.00
epoch  3: errors=4, w=[-0.2  0. ], b=0.00
epoch  4: errors=4, w=[-0.2  0. ], b=0.00
epoch  5: errors=4, w=[-0.2  0. ], b=0.00
epoch  6: errors=4, w=[-0.2  0. ], b=0.00
epoch  7: errors=4, w=[-0.2  0. ], b=0.00
epoch  8: errors=4, w=[-0.2  0. ], b=0.00
epoch  9: errors=4, w=[-0.2  0. ], b=0.00
epoch 10: errors=4, w=[-0.2  0. ], b=0.00
Số lỗi không về 0 ổn định vì XOR không khả tách tuyến tính.
